In [0]:
%sql
CREATE CATALOG IF NOT EXISTS uc_dev_snt_fdn

In [0]:
%sql

CREATE SCHEMA IF NOT EXISTS uc_dev_snt_fdn.config;

In [0]:
%sql
USE CATALOG uc_dev_snt_fdn;
USE SCHEMA config;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS uc_dev_snt_fdn.config.ingestion_metadata_sample (
    table_id STRING NOT NULL,
    source_system STRING,
    source_schema STRING,
    source_table STRING,
    reporting_unit_id STRING,

    -- Layer 0
    layer0_target_catalog STRING,
    layer0_target_schema STRING,
    layer0_target_table STRING,
    layer0_script_path STRING,
    layer0_load_type STRING,
    layer0_is_active BOOLEAN,

    -- Layer 1
    layer1_target_catalog STRING,
    layer1_target_schema STRING,
    layer1_target_table STRING,
    layer1_script_path STRING,
    layer1_load_type STRING,
    layer1_merge_keys STRING,
    layer1_column_mapping STRING,
    layer1_transformation_rules STRING,
    layer1_is_active BOOLEAN,

    -- Layer 2
    layer2_target_catalog STRING,
    layer2_target_schema STRING,
    layer2_target_table STRING,
    layer2_script_path STRING,
    layer2_load_type STRING,
    layer2_merge_keys STRING,
    layer2_column_mapping STRING,
    layer2_transformation_rules STRING,
    layer2_is_active BOOLEAN,

    partition_column STRING,
    watermark_column STRING,
    created_ts TIMESTAMP,
    updated_ts TIMESTAMP
)
USING DELTA;

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS uc_dev_snt_fdn.raw       COMMENT 'Layer 0 - raw landing, append only';
CREATE SCHEMA IF NOT EXISTS uc_dev_snt_fdn.curated   COMMENT 'Layer 1 - curated, MERGE upsert';
CREATE SCHEMA IF NOT EXISTS uc_dev_snt_fdn.gold      COMMENT 'Layer 2 - report ready, MERGE upsert';
CREATE SCHEMA IF NOT EXISTS uc_dev_snt_fdn.audit     COMMENT 'Centralized ingestion audit log';

CREATE SCHEMA IF NOT EXISTS uc_dev_snt_fdn.config    COMMENT 'Framework metadata';

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS uc_dev_snt_fdn.raw.source_files
    COMMENT 'Landing zone for POC source files, mimics the Snowflake extract drop location';

In [0]:
%sql

CREATE TABLE IF NOT EXISTS uc_dev_snt_fdn.audit.ingestion_audit_log (

    -- Identity
    audit_id            STRING      NOT NULL COMMENT 'Surrogate key (UUID) for this audit row',
    job_run_id          STRING               COMMENT 'Databricks Job Run ID - correlates all tasks in one workflow execution',
    pipeline_name       STRING               COMMENT 'Workflow/job name, e.g. gdo_master_workflow',
    notebook_name       STRING               COMMENT 'Notebook/script that executed (orchestrator.py)',

    -- What was processed
    table_name          STRING               COMMENT 'Business table, e.g. DIM_PRODUCT',
    layer                STRING              COMMENT 'Layer: 0, 1, or 2',
    source_system         STRING             COMMENT 'SNOWFLAKE in prod, FILE for this POC',
    reporting_unit_id      STRING            COMMENT 'Nullable - populated when reporting units are isolated',

    -- Where it ran
    environment              STRING          COMMENT 'dev / qa / uat / prod',
    workspace_url             STRING         COMMENT 'Databricks workspace URL the run executed in',
    executed_by                STRING        COMMENT 'Service principal or user identity that triggered the run',

    -- Timing
    start_time                   TIMESTAMP,
    end_time                      TIMESTAMP,
    duration_seconds                DOUBLE,

    -- Outcome
    status                          STRING   COMMENT 'RUNNING, SUCCESS, REJECTED, SKIPPED',
    records_read                     BIGINT,
    records_inserted                  BIGINT,
    records_updated                    BIGINT,
    records_rejected                    BIGINT,
    error_message                        STRING,

    -- Housekeeping
    created_ts                             TIMESTAMP
)
USING DELTA
COMMENT 'Centralized ingestion audit log - one row per table/layer/run, spans raw/curated/gold'
TBLPROPERTIES (
    'delta.autoOptimize.optimizeWrite' = 'true',
    'delta.autoOptimize.autoCompact'   = 'true'
);

In [0]:
%sql
drop table uc_dev_snt_fdn.raw.dim_product_raw

In [0]:
%sql
CREATE TABLE IF NOT EXISTS uc_dev_snt_fdn.raw.dim_product_raw (
    PRODUCT_ID          STRING,
    DESCRIPTION         STRING,
    PRODUCT_GROUP       STRING,
    SUB_GROUP           STRING,
    SUB_SUB_GROUP       STRING,
    SUB_SUB_SUB_GROUP   STRING,
    COLORFLAVOR         STRING,
    FLAVOR_FAMILY       STRING,
    PACK               STRING,
    CONFIGURATION       STRING,
    BRAND               STRING,
    EDITION             STRING,
    SCREW               STRING,
    CYLINDER_QTY        STRING,
    GAS_EXCHANGE        STRING,
    -- technical / lineage columns, added by the L0 script, not the source
    source_file_name    STRING,
    ingestion_timestamp  TIMESTAMP,
    job_run_id            STRING
)
USING DELTA
COMMENT 'Layer 0 - raw landing for DIM_PRODUCT, append only';


In [0]:
%sql

-- =====================================================================
-- LAYER 1 (curated) - STTM/business transformations applied.
-- Placeholder logic for now: snake_case columns, type casts, trim,
-- dedup on PRODUCT_ID. Real STTM will only change the transformation
-- script content, never this DDL shape drastically -- column_mapping
-- in metadata drives the renames.
-- =====================================================================
CREATE TABLE IF NOT EXISTS uc_dev_snt_fdn.curated.dim_product_curated (
    product_id           STRING       NOT NULL,
    description           STRING,
    product_group           STRING,
    sub_group                STRING,
    sub_sub_group              STRING,
    sub_sub_sub_group            STRING,
    color_flavor                   STRING,
    flavor_family                    STRING,
    pack                               DECIMAL(18,5),
    configuration                        STRING,
    brand                                  STRING,
    edition                                 STRING,
    screw                                    STRING,
    cylinder_qty                              INT,
    gas_exchange                                BOOLEAN,
    curated_load_timestamp                       TIMESTAMP,
    job_run_id                                     STRING
)
USING DELTA
COMMENT 'Layer 1 - curated DIM_PRODUCT, MERGE upsert on product_id';

-- =====================================================================
-- LAYER 2 (gold / GDO report-ready) - final, consumed by Power BI.
-- Adds one derived business column (category path) as an example of
-- what "finalized" gold logic looks like -- pass-through otherwise.
-- =====================================================================
CREATE TABLE IF NOT EXISTS uc_dev_snt_fdn.gold.gdo_dim_product (
    product_id             STRING     NOT NULL,
    description             STRING,
    product_group             STRING,
    sub_group                  STRING,
    sub_sub_group                 STRING,
    sub_sub_sub_group               STRING,
    product_category_path             STRING COMMENT 'product_group > sub_group > ... concatenated',
    color_flavor                        STRING,
    flavor_family                         STRING,
    pack                                    DECIMAL(18,5),
    configuration                             STRING,
    brand                                       STRING,
    edition                                       STRING,
    screw                                           STRING,
    cylinder_qty                                      INT,
    gas_exchange                                        BOOLEAN,
    gold_load_timestamp                                   TIMESTAMP,
    job_run_id                                              STRING
)
USING DELTA
COMMENT 'Layer 2 - GDO report-ready DIM_PRODUCT, consumed by Power BI';

In [0]:
mkdir -p /Workspace/snowflake_databricks_poc/transformations/layer0 /Workspace/snowflake_databricks_poc/transformations/layer1 /Workspace/snowflake_databricks_poc/transformations/layer2

In [0]:
with open('/Workspace/snowflake_databricks_poc/transformations/layer0/dim_product_l0.py', 'w') as f:
    f.write('''"""
transformations/layer0/dim_product_l0.py

Layer 0 (raw landing) for DIM_PRODUCT. Reads the source file from a Unity
Catalog Volume -- this is the ONLY thing that changes when you move to the
real project: swap this read for a Snowflake JDBC read. Everything below
the read (lineage columns, return shape) stays identical.
"""

from pyspark.sql.functions import current_timestamp, lit, input_file_name


def run(spark, metadata_row):
    """
    metadata_row.source_table holds the full Volume path for this POC
    (set in the metadata INSERT statement). In production this would
    instead be the Snowflake table name, and the read below would be
    a Snowflake JDBC read instead of spark.read.csv().
    """
    source_path = metadata_row["source_table"]  # e.g. /Volumes/.../dim_product_sample.csv

    df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "false")   # keep everything STRING at raw layer
        .csv(source_path)
    )

    df = (
        df
        .withColumn("source_file_name", input_file_name())
        .withColumn("ingestion_timestamp", current_timestamp())
        .withColumn("job_run_id", lit(metadata_row["table_id"]))  # replaced with real job_run_id by orchestrator context if needed
    )

    return df
''')

In [0]:
with open('/Workspace/snowflake_databricks_poc/transformations/layer1/dim_product_l1.py', 'w') as f:
    f.write('''"""
transformations/layer1/dim_product_l1.py

Layer 1 (curated) for DIM_PRODUCT. Placeholder STTM logic: rename to
snake_case, trim strings, cast types, dedup on product_id. When the real
STTM arrives, only the column_mapping / cast rules below change -- the
orchestrator and audit framework never need to be touched.
"""

from pyspark.sql.functions import (
    col, trim, current_timestamp, row_number, upper, when
)
from pyspark.sql.window import Window


def run(spark, metadata_row):
    raw_table = f"{metadata_row['layer0_target_catalog']}.{metadata_row['layer0_target_schema']}.{metadata_row['layer0_target_table']}"
    df = spark.table(raw_table)

    df = (
        df
        .select(
            trim(col("PRODUCT_ID")).alias("product_id"),
            trim(col("DESCRIPTION")).alias("description"),
            trim(col("PRODUCT_GROUP")).alias("product_group"),
            trim(col("SUB_GROUP")).alias("sub_group"),
            trim(col("SUB_SUB_GROUP")).alias("sub_sub_group"),
            trim(col("SUB_SUB_SUB_GROUP")).alias("sub_sub_sub_group"),
            trim(col("COLORFLAVOR")).alias("color_flavor"),
            trim(col("FLAVOR_FAMILY")).alias("flavor_family"),
            col("PACK").cast("decimal(18,5)").alias("pack"),
            trim(col("CONFIGURATION")).alias("configuration"),
            trim(col("BRAND")).alias("brand"),
            trim(col("EDITION")).alias("edition"),
            trim(col("SCREW")).alias("screw"),
            when(col("CYLINDER_QTY") == "", None)
                .otherwise(col("CYLINDER_QTY")).cast("int").alias("cylinder_qty"),
            (upper(trim(col("GAS_EXCHANGE"))) == "Y").alias("gas_exchange"),
        )
        .withColumn("curated_load_timestamp", current_timestamp())
        .withColumn("job_run_id", col("product_id"))  # placeholder, real job_run_id wired via widget in production
    )

    # Dedup: keep the most recently ingested row per product_id
    window = Window.partitionBy("product_id").orderBy(col("curated_load_timestamp").desc())
    df = (
        df.withColumn("_rn", row_number().over(window))
        .filter(col("_rn") == 1)
        .drop("_rn")
    )

    return df
''')

In [0]:
with open('/Workspace/snowflake_databricks_poc/transformations/layer2/dim_product_l2.py', 'w') as f:
    f.write('''"""
transformations/layer2/dim_product_l2.py

Layer 2 (GDO report-ready) for DIM_PRODUCT. Reads curated, adds one
derived business column (category path) as an example of finalization
logic, and writes gold_load_timestamp for lineage.
"""

from pyspark.sql.functions import col, concat_ws, current_timestamp


def run(spark, metadata_row):
    curated_table = f"{metadata_row['layer1_target_catalog']}.{metadata_row['layer1_target_schema']}.{metadata_row['layer1_target_table']}"
    df = spark.table(curated_table)

    df = (
        df
        .withColumn(
            "product_category_path",
            concat_ws(" > ", col("product_group"), col("sub_group"),
                      col("sub_sub_group"), col("sub_sub_sub_group"))
        )
        .withColumn("gold_load_timestamp", current_timestamp())
    )

    return df
''')

In [0]:
mkdir -p /Workspace/Shared/snowflake_databricks_poc/sql

In [0]:
with open('/Workspace/Shared/snowflake_databricks_poc/sql/04_metadata_insert_and_updates.sql', 'w') as f:
    f.write('''
-- =====================================================================
-- INSERT: one row for DIM_PRODUCT covering all three layers.
-- Replace <YOUR_GIT_FOLDER_PATH> with your actual path, e.g.:
--   /Workspace/Users/you@example.com/gdo-databricks-poc
-- =====================================================================
INSERT INTO uc_dev_snt_fdn.config.ingestion_metadata (
    table_id, source_system, source_schema, source_table, reporting_unit_id,

    layer0_target_catalog, layer0_target_schema, layer0_target_table,
    layer0_script_path, layer0_load_type, layer0_is_active,

    layer1_target_catalog, layer1_target_schema, layer1_target_table,
    layer1_script_path, layer1_load_type, layer1_merge_keys,
    layer1_column_mapping, layer1_transformation_rules, layer1_is_active,

    layer2_target_catalog, layer2_target_schema, layer2_target_table,
    layer2_script_path, layer2_load_type, layer2_merge_keys,
    layer2_column_mapping, layer2_transformation_rules, layer2_is_active,

    partition_column, watermark_column, created_ts, updated_ts
)
VALUES (
    'DIM_PRODUCT', 'FILE', 'source_files',
    '/Volumes/uc_dev_snt_fdn/raw/source_files/dim_product_sample.csv', NULL,

    'uc_dev_snt_fdn', 'raw', 'dim_product_raw',
    '/Workspace/Shared/snowflake_databricks_poc/transformations/layer0/dim_product_l0.py', 'APPEND', true,

    'uc_dev_snt_fdn', 'curated', 'dim_product_curated',
    '/Workspace/Shared/snowflake_databricks_poc/transformations/layer1/dim_product_l1.py', 'MERGE', 'product_id',
    NULL, NULL, true,

    'uc_dev_snt_fdn', 'gold', 'gdo_dim_product',
    '/Workspace/Shared/snowflake_databricks_poc/transformations/layer2/dim_product_l2.py', 'MERGE', 'product_id',
    NULL, NULL, true,

    NULL, NULL, current_timestamp(), current_timestamp()
);

-- =====================================================================
-- Example UPDATEs you'll actually use during development
-- =====================================================================

-- Toggle Layer 2 off temporarily (e.g. gold logic not ready yet)
UPDATE uc_dev_snt_fdn.config.ingestion_metadata
SET layer2_is_active = false, updated_ts = current_timestamp()
WHERE table_id = 'DIM_PRODUCT';

-- Turn it back on
UPDATE uc_dev_snt_fdn.config.ingestion_metadata
SET layer2_is_active = true, updated_ts = current_timestamp()
WHERE table_id = 'DIM_PRODUCT';

-- Fix a script path after moving/renaming a file in your Git folder
UPDATE uc_dev_snt_fdn.config.ingestion_metadata
SET layer1_script_path = '<NEW_PATH>/dim_product_l1.py', updated_ts = current_timestamp()
WHERE table_id = 'DIM_PRODUCT';

-- Point source_table at the real Snowflake table name when you move off the POC
UPDATE uc_dev_snt_fdn.config.ingestion_metadata
SET source_system = 'SNOWFLAKE',
    source_schema = 'SALES',
    source_table = 'DIM_PRODUCT',
    updated_ts = current_timestamp()
WHERE table_id = 'DIM_PRODUCT';
EOF
echo "metadata insert/update queries written"
''')

In [0]:
%sql
INSERT INTO uc_dev_snt_fdn.config.ingestion_metadata_sample (
    table_id, source_system, source_schema, source_table, reporting_unit_id,

    layer0_target_catalog, layer0_target_schema, layer0_target_table,
    layer0_script_path, layer0_load_type, layer0_is_active,

    layer1_target_catalog, layer1_target_schema, layer1_target_table,
    layer1_script_path, layer1_load_type, layer1_merge_keys,
    layer1_column_mapping, layer1_transformation_rules, layer1_is_active,

    layer2_target_catalog, layer2_target_schema, layer2_target_table,
    layer2_script_path, layer2_load_type, layer2_merge_keys,
    layer2_column_mapping, layer2_transformation_rules, layer2_is_active,

    partition_column, watermark_column, created_ts, updated_ts
)
VALUES (
    'DIM_PRODUCT', 'FILE', 'source_files',
    '/Volumes/uc_dev_snt_fdn/raw/source_files/dim_product_sample.csv', NULL,

    'uc_dev_snt_fdn', 'raw', 'dim_product_raw',
    '/Workspace/Shared/snowflake_databricks_poc/transformations/layer0/dim_product_l0.py', 'APPEND', true,

    'uc_dev_snt_fdn', 'curated', 'dim_product_curated',
    '/Workspace/Shared/snowflake_databricks_poc/transformations/layer1/dim_product_l1.py', 'MERGE', 'product_id',
    NULL, NULL, true,

    'uc_dev_snt_fdn', 'gold', 'gdo_dim_product',
    '/Workspace/Shared/snowflake_databricks_poc/transformations/layer2/dim_product_l2.py', 'MERGE', 'product_id',
    NULL, NULL, true,

    NULL, NULL, current_timestamp(), current_timestamp()
);

In [0]:
%sql
select * from uc_dev_snt_fdn.config.ingestion_metadata_sample

In [0]:
%sql

 UPDATE uc_dev_snt_fdn.config.ingestion_metadata_sample
SET reporting_unit_id = "10", updated_ts = current_timestamp()
WHERE table_id = 'DIM_PRODUCT';

In [0]:
%sql
-- LAYER 0 (raw)
CREATE TABLE IF NOT EXISTS uc_dev_snt_fdn.raw.dim_customer_raw (
    INDEXS              STRING,
    CUSTOMER_ID          STRING,
    CUSTOMER_NAME        STRING,
    CUSTOMER_GROUP       STRING,
    CUSTOMER_GROUP2      STRING,
    CUSTOMER_GROUP3      STRING,
    CUSTOMERPARENTID     STRING,
    PARENTNAME           STRING,
    PATHNAME             STRING,
    DEPTH                STRING,
    BT_CITY              STRING,
    BT_PROVINCE          STRING,
    PROVINCE             STRING,
    COUNTRY              STRING,
    source_file_name     STRING,
    ingestion_timestamp  TIMESTAMP,
    job_run_id           STRING
)
USING DELTA
COMMENT 'Layer 0 - raw landing for DIM_CUSTOMER';

-- LAYER 1 (curated)
CREATE TABLE IF NOT EXISTS uc_dev_snt_fdn.curated.dim_customer_curated (
    customer_id           STRING NOT NULL,
    customer_name          STRING,
    customer_group           STRING,
    customer_group2            STRING,
    customer_group3              STRING,
    customer_parent_id             STRING,
    parent_name                      STRING,
    path_name                          STRING,
    depth                                BIGINT,
    bt_city                                STRING,
    bt_province                              STRING,
    province                                   STRING,
    country                                      STRING,
    curated_load_timestamp                         TIMESTAMP,
    job_run_id                                       STRING
)
USING DELTA
COMMENT 'Layer 1 - curated DIM_CUSTOMER, MERGE upsert on customer_id';

-- LAYER 2 (gold)
CREATE TABLE IF NOT EXISTS uc_dev_snt_fdn.gold.gdo_dim_customer (
    customer_id           STRING NOT NULL,
    customer_name          STRING,
    customer_group           STRING,
    customer_group2            STRING,
    customer_group3              STRING,
    customer_parent_id             STRING,
    parent_name                      STRING,
    customer_hierarchy_path            STRING COMMENT 'parent_name > customer_name',
    depth                                 BIGINT,
    bt_city                                 STRING,
    bt_province                               STRING,
    province                                    STRING,
    country                                       STRING,
    gold_load_timestamp                             TIMESTAMP,
    job_run_id                                        STRING
)
USING DELTA
COMMENT 'Layer 2 - GDO report-ready DIM_CUSTOMER';


In [0]:
%sql
-- LAYER 0 (raw)
CREATE TABLE IF NOT EXISTS uc_dev_snt_fdn.raw.dim_entities_raw (
    ENTITY_CODE               STRING,
    ENTITY_NAME               STRING,
    ENTITY_GROUP              STRING,
    ENTITY_GROUP_SHORT_NAME   STRING,
    DEFAULT_MARKET            STRING,
    COMPANY                   STRING,
    DWH_UPDATE_DATE           STRING,
    source_file_name          STRING,
    ingestion_timestamp       TIMESTAMP,
    job_run_id                STRING
)
USING DELTA
COMMENT 'Layer 0 - raw landing for DIM_ENTITIES';

-- LAYER 1 (curated)
CREATE TABLE IF NOT EXISTS uc_dev_snt_fdn.curated.dim_entities_curated (
    entity_code               STRING NOT NULL,
    entity_name                STRING,
    entity_group                 STRING,
    entity_group_short_name        STRING,
    default_market                   STRING,
    company                            STRING,
    dwh_update_date                      TIMESTAMP,
    curated_load_timestamp                 TIMESTAMP,
    job_run_id                               STRING
)
USING DELTA
COMMENT 'Layer 1 - curated DIM_ENTITIES, MERGE upsert on entity_code';

-- LAYER 2 (gold)
CREATE TABLE IF NOT EXISTS uc_dev_snt_fdn.gold.gdo_dim_entities (
    entity_code               STRING NOT NULL,
    entity_name                STRING,
    entity_group                 STRING,
    entity_group_short_name        STRING,
    default_market                   STRING,
    company                            STRING,
    dwh_update_date                      TIMESTAMP,
    gold_load_timestamp                    TIMESTAMP,
    job_run_id                               STRING
)
USING DELTA
COMMENT 'Layer 2 - GDO report-ready DIM_ENTITIES';


In [0]:
%sql
INSERT INTO uc_dev_snt_fdn.config.ingestion_metadata_sample (
    table_id, source_system, source_schema, source_table, reporting_unit_id, table_group,
    layer0_target_catalog, layer0_target_schema, layer0_target_table, layer0_script_path, layer0_load_type, layer0_is_active,
    layer1_target_catalog, layer1_target_schema, layer1_target_table, layer1_script_path, layer1_load_type, layer1_merge_keys,
    layer1_column_mapping, layer1_transformation_rules, layer1_is_active,
    layer2_target_catalog, layer2_target_schema, layer2_target_table, layer2_script_path, layer2_load_type, layer2_merge_keys,
    layer2_column_mapping, layer2_transformation_rules, layer2_is_active,
    partition_column, watermark_column, created_ts, updated_ts
)
VALUES
(
    'DIM_CUSTOMER', 'FILE', 'source_files',
    '/Volumes/uc_dev_snt_fdn/raw/source_files/dim_customer_sample.csv', NULL, 'DIMENSION',
    'uc_dev_snt_fdn', 'raw', 'dim_customer_raw',
    '/Workspace/Shared/snowflake_databricks_poc/transformations/layer0/dim_customer_l0.py', 'APPEND', true,
    'uc_dev_snt_fdn', 'curated', 'dim_customer_curated',
    '/Workspace/Shared/snowflake_databricks_poc/transformations/layer1/dim_customer_l1.py', 'MERGE', 'customer_id',
    NULL, NULL, true,
    'uc_dev_snt_fdn', 'gold', 'gdo_dim_customer',
    '/Workspace/Shared/snowflake_databricks_poc/transformations/layer2/dim_customer_l2.py', 'MERGE', 'customer_id',
    NULL, NULL, true,
    NULL, NULL, current_timestamp(), current_timestamp()
),
(
    'DIM_ENTITIES', 'FILE', 'source_files',
    '/Volumes/uc_dev_snt_fdn/raw/source_files/dim_entities_sample.csv', NULL, 'DIMENSION',
    'uc_dev_snt_fdn', 'raw', 'dim_entities_raw',
    '/Workspace/Shared/snowflake_databricks_poc/transformations/layer0/dim_entities_l0.py', 'APPEND', true,
    'uc_dev_snt_fdn', 'curated', 'dim_entities_curated',
    '/Workspace/Shared/snowflake_databricks_poc/transformations/layer1/dim_entities_l1.py', 'MERGE', 'entity_code',
    NULL, NULL, true,
    'uc_dev_snt_fdn', 'gold', 'gdo_dim_entities',
    '/Workspace/Shared/snowflake_databricks_poc/transformations/layer2/dim_entities_l2.py', 'MERGE', 'entity_code',
    NULL, NULL, true,
    NULL, NULL, current_timestamp(), current_timestamp()
);

In [0]:
%sql
ALTER TABLE uc_dev_snt_fdn.config.ingestion_metadata_sample
ADD COLUMN table_is_active BOOLEAN;

In [0]:
%sql
UPDATE uc_dev_snt_fdn.config.ingestion_metadata_sample
SET table_is_active = TRUE
WHERE table_is_active IS NULL;  

In [0]:
pip install databricks

In [0]:
%restart_python

In [0]:
databricks -v